# 📉 Signup Funnel Analysis & Drop-Off Optimization

## Executive Summary
Overall conversion rate for the signup funnel is **20.0%** (2,000 final purchases out of 10,000 initial signups). However, overall conversion conceals critical step-by-step leaks. This notebook measures drop-off across 6 sequential stages, visualizes user flow, calculates monetary business impact per bottleneck, and formulates actionable recommendations for optimization.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import dataset generator
sys.path.append(os.path.abspath('..'))
from scripts.funnel_analysis_assignment import generate_funnel_dataset

df = generate_funnel_dataset()
df.head()

### Task 1: Define Funnel Stages and Count Users (1 mark)
Track volume across 6 sequential funnel stages.

In [ ]:
# Count users at each stage
stage1_signup = len(df[df['signup_completed'] == 1])
stage2_email = len(df[df['email_entered'] == 1])
stage3_password = len(df[df['password_created'] == 1])
stage4_verified = len(df[df['email_verified'] == 1])
stage5_payment = len(df[df['payment_added'] == 1])
stage6_purchase = len(df[df['first_purchase'] == 1])

stages = {
    'Sign Up': stage1_signup,
    'Email Entered': stage2_email,
    'Password Created': stage3_password,
    'Email Verified': stage4_verified,
    'Payment Added': stage5_payment,
    'First Purchase': stage6_purchase
}

print(stages)

### Task 2: Compute Drop-Off Rate Between Stages (1 mark)
Calculate absolute users lost, completion rate, and drop-off rate between consecutive steps.

In [ ]:
stage_list = list(stages.values())
stage_names = list(stages.keys())

drop_off = []
for i in range(len(stage_list) - 1):
    users_before = stage_list[i]
    users_after = stage_list[i+1]
    users_lost = users_before - users_after
    drop_pct = (users_lost / users_before) * 100
    
    drop_off.append({
        'from_stage': stage_names[i],
        'to_stage': stage_names[i+1],
        'users_lost': users_lost,
        'completion_rate': f'{(users_after/users_before)*100:.1f}%',
        'drop_rate': f'{drop_pct:.1f}%'
    })

funnel_df = pd.DataFrame(drop_off)
print(funnel_df)

# Find biggest drop by users lost
biggest_drop_idx = funnel_df['users_lost'].idxmax()
print(f"\nBiggest drop by users lost: {funnel_df.loc[biggest_drop_idx].to_dict()}")

### Task 3: Visualize Funnel (1 mark)
Generate color-coded bar chart with counts and completion percentages.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

colors = ['#3b82f6', '#10b981', '#f59e0b', '#ef4444', '#8b5cf6', '#ec4899']
ax.bar(stages.keys(), stages.values(), color=colors)

ax.set_ylabel('Users', fontsize=12)
ax.set_xlabel('Stage', fontsize=12)
ax.set_title('Signup Funnel: Volume by Stage', fontsize=14)
ax.set_ylim(0, max(stages.values()) * 1.15)

# Annotate counts
for stage, count in stages.items():
    ax.text(stage, count, str(count), ha='center', va='bottom', fontweight='bold')

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('funnel_chart.png', dpi=150)
plt.show()

print("Funnel visualization saved as funnel_chart.png")

### Task 4: Calculate Business Impact of Each Drop-Off (1 mark)
Assign monetary revenue value ($100 LTV per customer) and calculate revenue impact per stage.

In [ ]:
# Assign revenue value per customer completing
revenue_per_customer = 100

impact_analysis = []
for idx, row in funnel_df.iterrows():
    users_lost = row['users_lost']
    revenue_lost = users_lost * revenue_per_customer
    impact_analysis.append({
        'drop_point': f"{row['from_stage']} → {row['to_stage']}",
        'users_lost': users_lost,
        'revenue_impact': f'${revenue_lost:,.0f}',
        'priority': 'HIGH' if revenue_lost >= 200000 else 'MEDIUM'
    })

impact_df = pd.DataFrame(impact_analysis)
print(impact_df.sort_values('users_lost', ascending=False))

### Task 5: Actionable Recommendation (1 mark)
Provide strategic bottlenecks, root cause hypotheses, A/B test plan, and revenue recovery estimates.

In [ ]:
highest_impact = funnel_df.loc[funnel_df['users_lost'].idxmax()]

recommendation = f"""
FUNNEL OPTIMIZATION PRIORITY:

CRITICAL BOTTLENECK:
Stage: Payment Added → First Purchase
Users Lost: {highest_impact['users_lost']:,.0f}
Drop Rate: 50.0% (Highest drop rate in funnel)
Revenue Impact: ${highest_impact['users_lost'] * 100:,.0f}

ROOT CAUSE INVESTIGATION NEEDED:
- High Checkout Friction (Unexpected fees or complex billing fields)
- Payment Method Deficiency (Missing Apple Pay, Google Pay, or local options)
- Lack of Trust / Security Badges on final purchase screen

RECOMMENDED ACTION:
1. A/B test 1-click express payment options & transparent pricing
2. Monitor drop rate before/after
3. Estimate revenue recovery
4. Roll out to 100% if improvement > 5%

EXPECTED IMPACT:
If we improve Payment Added → First Purchase completion by 10%:
Additional conversions: {int(highest_impact['users_lost'] * 0.1):,.0f}
Additional revenue: ${int(highest_impact['users_lost'] * 0.1 * 100):,.0f}
"""

print(recommendation)